In [0]:
from pyspark.sql.functions import round, lit, col, when
from datetime import datetime

In [0]:
cur_businessdate = datetime.now().strftime('%Y%m01')
df_gold = (
    spark.read.table('mobills.silver.orcamentos')
    .withColumn('consumido', round(col('efetivado') + col('previsto'), 2))
    .withColumn('base_saldo', round(col('planejado') - col('consumido'), 2))
    .withColumn(
        'saldo',
        when(col('_businessdate') < cur_businessdate, lit(0))
        .when(col('base_saldo') < 0, lit(0))
        .otherwise(col('base_saldo'))
    )
    .select(
        'data', 'categoria', 'subcategoria', 'planejado', 'consumido', 'saldo'
    )
)

(
    df_gold
    .write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable("mobills.gold.orcamentos")
)